# Notebook 2: Match-Sensitive LCBS Algorithm — O(M log²M) Implementation

## Why "Match-Sensitive"?

The baseline algorithm (Notebook 1) always takes **Θ(nm)** time regardless of the input structure. This notebook implements the **instance-sensitive** algorithm whose cost depends on:

$$M = |V| = |\{(i,j) : A[i] = B[j]\}| \quad \text{(number of matches)}$$

**Total time:** $O(n + m + M \log^2 M)$

### When does it beat the baseline?

When $M \log^2 M \ll nm$, i.e. the **sparse match regime** — large alphabets produce few coincidences:

| Alphabet size $\alpha$ | Expected $M$ | Algorithm choice |
|---|---|---|
| Small ($\alpha \ll n$) | $\Theta(nm/\alpha) \approx nm$ | Baseline wins |
| Large ($\alpha \sim n$) | $O(m)$ expected | **Match-sensitive wins** |

## Paper Reference

> Md. Tanzeem Rahat and Md. Manzurul Hasan. *The Longest Common Bitonic Subsequence: A Match-Sensitive Dynamic Programming Approach*. arXiv:2511.08958v2, 2026.

**Section 4, Algorithm 3** — implemented here in full.

## Key Idea

Instead of scanning all $nm$ cells, we work exclusively on the **match poset** $V$. A match $u = (i, j)$ **dominates** $v = (i', j')$ (written $u \prec v$) when $i < i'$ and $j < j'$ — both indices strictly smaller.

The INC/DEC DP recurrences are then solved by **2D dominance range-maximum queries** using a 2D Binary Indexed Tree (Fenwick tree), giving $O(\log^2 M)$ per match.

In [1]:
import numpy as np
import time
import random
import os
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Tuple, Dict, Optional
from sortedcontainers import SortedList

## Step 1: Building the Match Set V (Section 4.1)

The naive approach scans all $nm$ pairs — $O(nm)$ time. Instead we preprocess $B$ into a position dictionary:

$$\text{PosB}[x] = \{j : B[j] = x\} \quad \text{built in } O(m)$$

Then for each $i$, we enumerate only $j \in \text{PosB}[A[i]]$ — no wasted work. Total time: **O(n + m + M)**.

The output is sorted lexicographically by $(i, j)$, which is the **topological order** of the dominance poset: if $i < i'$ then $(i,j)$ is processed before $(i',j')$ in the forward pass, ensuring all predecessors are ready when we compute $\text{INC}[v]$.

In [2]:
def build_match_set(A: List, B: List) -> List[Tuple]:
    """
    Build V = {(i,j) : A[i]=B[j]} in O(n+m+M) time.

    Uses a position dictionary: PosB[val] = sorted list of j-indices
    where B[j] = val. Then for each i, enumerate j in PosB[A[i]].

    Returns: list of (i,j) tuples sorted lexicographically (i asc, j asc).
    Paper: Section 4.1, "Building the match set in O(m+M) time."
    """
    from collections import defaultdict
    pos_B = defaultdict(list)
    for j, val in enumerate(B):
        pos_B[val].append(j)

    matches = []
    for i, val in enumerate(A):
        for j in pos_B[val]:       # pos_B[val] already in ascending j order
            matches.append((i, j))

    matches.sort()                 # lex order: (i asc, j asc)
    return matches

## Step 2: Coordinate Compression (Section 4.1)

The dominance oracle uses integer coordinates. Raw $j$-indices can be up to $m-1$ and raw values up to any integer — both potentially large. Coordinate compression maps them to tight integer ranks:

$$r_J(v) \in \{1, \ldots, J\}, \quad J \leq M \qquad \text{(compressed } j\text{-rank)}$$
$$r_V(v) \in \{1, \ldots, R\}, \quad R \leq M \qquad \text{(compressed value rank)}$$

**Critical property:** strict inequalities are preserved under compression:
$$j(u) < j(v) \iff r_J(u) \leq r_J(v) - 1$$
$$\text{val}(u) < \text{val}(v) \iff r_V(u) \leq r_V(v) - 1$$

This lets the Fenwick tree work on indices $1..J$ and $1..R$ instead of raw values, keeping the BIT size at $O(M)$.

In [3]:
def coordinate_compress(matches: List[Tuple], A: List, B: List) -> Dict:
    """
    Compress coordinates for the dominance oracle.

    rJ[v] = rank of j(v) among all distinct j-coords in matches (1-indexed)
    rV[v] = rank of val(v) among all distinct values in matches (1-indexed)

    Paper: Section 4.1, "Coordinate compression."
    """
    if not matches:
        return {'rJ': {}, 'rV': {}, 'J': 0, 'R': 0}

    j_vals = sorted(set(j for (i, j) in matches))
    v_vals = sorted(set(A[i] for (i, j) in matches))

    j_rank = {j: r + 1 for r, j in enumerate(j_vals)}
    v_rank = {v: r + 1 for r, v in enumerate(v_vals)}

    rJ = {(i, j): j_rank[j]    for (i, j) in matches}
    rV = {(i, j): v_rank[A[i]] for (i, j) in matches}
    J  = len(j_vals)
    R  = len(v_vals)

    return {'rJ': rJ, 'rV': rV, 'J': J, 'R': R}

## Step 3: 2D Dominance Maximum Oracle (Lemma 4)

**What we need:** At any point during the forward pass, given a new match $v$, we must find:

$$\text{Query}(X, Y) = \max\{\text{INC}[u] : r_J(u) \leq X,\; r_V(u) \leq Y\}$$

This is a **2D orthogonal range maximum query** with point updates.

**Implementation:** A **2D Binary Indexed Tree (Fenwick tree)** — the standard practical realisation of Lemma 4's "2D range tree" guarantee:

| Operation | Time | Space |
|---|---|---|
| `update(x, y, key, id)` | $O(\log J \cdot \log R) = O(\log^2 M)$ | — |
| `query(X, Y)` | $O(\log J \cdot \log R) = O(\log^2 M)$ | — |
| Total over $M$ calls | $O(M \log^2 M)$ | $O(M \log M)$ |

**Paper:** Lemma 4, Section 4.3 — the oracle is used twice: once for the forward pass (INC) and once for the backward pass (DEC with mirrored $x$-coordinate).

In [4]:
class DominanceMaxOracle:
    """
    2D dominance maximum oracle via a sparse 2D Binary Indexed Tree.

    Supports:
      update(x, y, key, ident) — record key at point (x, y)
      query(X, Y)              — return (max_key, argmax_id) for x<=X, y<=Y

    Time per operation: O(log(max_x) * log(max_y)) = O(log²M)
    Space: O(M log M) amortised (sparse dict — only touched cells stored)

    Paper: Lemma 4, Section 4.3.
    """

    def __init__(self, max_x: int, max_y: int):
        self.max_x = max_x
        self.max_y = max_y
        self.bit: Dict[Tuple[int,int], Tuple[int, object]] = {}  # (x,y)->(key,id)

    def _update_cell(self, x: int, y: int, key: int, ident):
        cell = self.bit.get((x, y))
        if cell is None or key > cell[0]:
            self.bit[(x, y)] = (key, ident)

    def update(self, x: int, y: int, key: int, ident):
        """Insert/update point (x,y) with key and identifier."""
        xi = x
        while xi <= self.max_x:
            yi = y
            while yi <= self.max_y:
                self._update_cell(xi, yi, key, ident)
                yi += yi & (-yi)
            xi += xi & (-xi)

    def query(self, X: int, Y: int) -> Tuple[int, Optional[object]]:
        """Return (max_key, argmax_id) over all points with x<=X, y<=Y."""
        best_key = 0
        best_id  = None
        xi = X
        while xi > 0:
            yi = Y
            while yi > 0:
                cell = self.bit.get((xi, yi))
                if cell is not None and cell[0] > best_key:
                    best_key, best_id = cell
                yi -= yi & (-yi)
            xi -= xi & (-xi)
        return best_key, best_id

    def reset(self):
        """Clear the oracle for reuse."""
        self.bit.clear()

## Step 4: Forward Pass — Rising DP (Equation 2)

**Recurrence (Eq. 2 from paper):**

$$\text{INC}[v] = 1 + \max\{\text{INC}[u] : u \prec v,\; \text{val}(u) < \text{val}(v)\}, \quad \max \emptyset = 0$$

The dominance constraint $u \prec v$ means $i(u) < i(v)$ **and** $j(u) < j(v)$. In compressed coordinates:

$$\text{Query}(T_{\text{inc}},\; r_J(v)-1,\; r_V(v)-1)$$

**Why lexicographic scan guarantees correctness:** Matches are processed in order of increasing $(i, j)$. When we process $v = (i', j')$, every $u = (i, j)$ with $i < i'$ has already been inserted into $T_{\text{inc}}$. Among those, the query filters $j(u) \leq r_J(v)-1$ (i.e. $j < j'$) and $\text{val}(u) \leq r_V(v)-1$ (i.e. $\text{val}(u) < \text{val}(v)$) — exactly the predecessor set.

## Step 5: Backward Pass — Falling DP (Equation 3)

**Recurrence (Eq. 3 from paper):**

$$\text{DEC}[v] = 1 + \max\{\text{DEC}[w] : v \prec w,\; \text{val}(w) < \text{val}(v)\}, \quad \max \emptyset = 0$$

We need $r_J(w) > r_J(v)$ — a **suffix** query in $j$, not a prefix. To convert it into a prefix query (which the BIT naturally supports), we apply the **mirrored $j$-rank trick**:

$$\hat{x}(v) = J - r_J(v) + 1$$

Under this mirror, larger $r_J(w)$ maps to smaller $\hat{x}(w)$, so the constraint $r_J(w) > r_J(v)$ becomes $\hat{x}(w) < \hat{x}(v)$, i.e. $\hat{x}(w) \leq \hat{x}(v) - 1$ — a prefix query again.

Processing matches in **reverse lexicographic order** ensures that when we handle $v$, all successors $w \succ v$ (with larger $i$ and $j$) are already in $T_{\text{dec}}$.

In [5]:
def solve_lcbs_match_sensitive(A: List, B: List) -> Dict:
    """
    Match-Sensitive LCBS Algorithm — Algorithm 3 from Rahat & Hasan (2026).

    Steps:
      1. Build match set V in O(n+m+M)
      2. Coordinate compression in O(M log M)
      3. Forward pass (rising DP)  via dominance oracle: O(M log^2 M)
      4. Backward pass (falling DP) via mirrored oracle: O(M log^2 M)
      5. Peak scan in O(M)
      6. Reconstruction via pred/succ pointers in O(l)

    Total: O(n+m+M log^2 M) time, O(M log M) space.
    Paper: Section 4, Algorithm 3, Theorem 4, Theorem 5.

    Key correctness fix: matches sharing the same row-i must be queried
    BEFORE any of them update the BIT, to avoid a same-row match u=(i,j1)
    falsely contributing to INC[v=(i,j2)] with j1<j2 (u does not dominate v
    because they share the same i-index).
    """
    from collections import defaultdict
    start = time.perf_counter()
    n, m = len(A), len(B)

    # Step 1: Build match set
    matches = build_match_set(A, B)
    M = len(matches)

    if M == 0:
        return {'sequence': [], 'length': 0, 'peak_match': None,
                'peak_value': None, 'INC': {}, 'DEC': {},
                'num_matches': 0, 'time_ms': 0.0}

    # Step 2: Coordinate compression
    cc = coordinate_compress(matches, A, B)
    rJ, rV = cc['rJ'], cc['rV']
    J,  R  = cc['J'],  cc['R']

    # Group matches by row for correct same-i handling
    by_row = defaultdict(list)
    for v in matches: by_row[v[0]].append(v[1])

    # Step 3: Forward pass - Rising DP
    # BIT axes: x = rJ(j), y = rV(val)
    # Process rows in ascending order; within each row: query all, then update all.
    INC  = {}
    pred = {}
    T_inc = DominanceMaxOracle(J, R)

    for i in sorted(by_row):
        # Query phase (BIT has only rows < i)
        row_q = {}
        for j in by_row[i]:
            v = (i, j)
            x, y = rJ[v], rV[v]
            bk, bi = T_inc.query(x - 1, y - 1) if (x > 1 and y > 1) else (0, None)
            row_q[j] = (bk + 1, bi)
        # Update phase
        for j in by_row[i]:
            v = (i, j)
            val, pv = row_q[j]
            INC[v] = val; pred[v] = pv
            T_inc.update(rJ[v], rV[v], val, v)

    # Step 4: Backward pass - Falling DP
    # DEC[v] = 1 + max{DEC[w] : i_w > i_v, j_w > j_v, val(w) < val(v)}
    # Mirror j-rank: x_hat = J - rJ(v) + 1  (large j -> small x_hat, prefix becomes suffix)
    # Process rows in descending order; same query-before-update discipline.
    DEC  = {}
    succ = {}
    T_dec = DominanceMaxOracle(J, R)

    for i in sorted(by_row, reverse=True):
        row_q2 = {}
        for j in by_row[i]:
            v = (i, j)
            x_hat = J - rJ[v] + 1
            y     = rV[v]
            bk, bi = T_dec.query(x_hat - 1, y - 1) if (x_hat > 1 and y > 1) else (0, None)
            row_q2[j] = (bk + 1, bi)
        for j in by_row[i]:
            v = (i, j)
            val, sv = row_q2[j]
            DEC[v] = val; succ[v] = sv
            T_dec.update(J - rJ[v] + 1, rV[v], val, v)

    # Step 5: Peak scan
    best_score = 0
    peak = None
    for v in matches:
        score = INC[v] + DEC[v] - 1
        if score > best_score:
            best_score = score
            peak = v

    # Step 6: Reconstruction
    if peak is None:
        seq = []
    else:
        inc = []
        cur = peak
        while cur is not None:
            inc.append(A[cur[0]]); cur = pred.get(cur)
        inc.reverse()
        dec = []
        cur = succ.get(peak)
        while cur is not None:
            dec.append(A[cur[0]]); cur = succ.get(cur)
        seq = inc + dec

    elapsed_ms = (time.perf_counter() - start) * 1000
    return {
        'sequence': seq, 'length': best_score, 'peak_match': peak,
        'peak_value': A[peak[0]] if peak else None,
        'INC': INC, 'DEC': DEC, 'num_matches': M,
        'J': J, 'R': R, 'time_ms': elapsed_ms
    }


## ✅ Test 1: Paper's Motivating Example — Both Algorithms Compared

We also inline the baseline algorithm here so this notebook is fully self-contained.

In [6]:
# ── Inline baseline (Algorithm 1 + 2 from Notebook 1) for self-contained comparison ──
def _lcis_lengths(A, B):
    n, m = len(A), len(B)
    dp, endI, endJ = [0]*m, [-1]*m, [-1]*m
    INC, pred = {}, {}
    for i in range(n):
        bestLen, bestEnd = 0, None
        for j in range(m):
            if A[i] == B[j]:
                cand = bestLen + 1
                INC[(i,j)] = cand
                pred[(i,j)] = bestEnd
                if cand > dp[j]:
                    dp[j], endI[j], endJ[j] = cand, i, j
            if B[j] < A[i] and dp[j] > bestLen:
                bestLen, bestEnd = dp[j], (endI[j], endJ[j])
    return INC, pred

def solve_lcbs_baseline(A, B):
    n, m = len(A), len(B)
    INC, pred = _lcis_lengths(A, B)
    if not INC:
        return {'sequence': [], 'length': 0, 'peak_match': None, 'peak_value': None,
                'INC': {}, 'DEC': {}, 'num_matches': 0, 'time_ms': 0.0}
    INC_rev, pred_rev = _lcis_lengths(A[::-1], B[::-1])
    DEC, succ = {}, {}
    for (i, j) in INC:
        ir, jr = n-1-i, m-1-j
        DEC[(i,j)] = INC_rev.get((ir, jr), 1)
        pr = pred_rev.get((ir, jr))
        succ[(i,j)] = None if pr is None else (n-1-pr[0], m-1-pr[1])
    best_score, peak = 0, None
    for match in INC:
        s = INC[match] + DEC.get(match, 1) - 1
        if s > best_score:
            best_score, peak = s, match
    if peak is None:
        seq = []
    else:
        inc = []
        cur = peak
        while cur is not None:
            inc.append(A[cur[0]]); cur = pred.get(cur)
        inc.reverse()
        dec = []
        cur = succ.get(peak)
        while cur is not None:
            dec.append(A[cur[0]]); cur = succ.get(cur)
        seq = inc + dec
    return {'sequence': seq, 'length': best_score, 'peak_match': peak,
            'peak_value': A[peak[0]] if peak else None,
            'INC': INC, 'DEC': DEC, 'num_matches': len(INC), 'time_ms': 0.0}

# ── Paper motivating example ──────────────────────────────────────────────
A = [2, 1, 3, 4, 6, 5, 4]
B = [1, 2, 3, 5, 6, 4]

r1 = solve_lcbs_baseline(A, B)
r2 = solve_lcbs_match_sensitive(A, B)

print("Paper example: A=[2,1,3,4,6,5,4], B=[1,2,3,5,6,4]")
print(f"Baseline result:        seq={r1['sequence']}, len={r1['length']}")
print(f"Match-sensitive result: seq={r2['sequence']}, len={r2['length']}")
print(f"M (match count):        {r2['num_matches']}")
print(f"J (distinct j-ranks):   {r2['J']},  R (distinct value-ranks): {r2['R']}")

status = "✅ BOTH AGREE" if r1['length'] == r2['length'] else "❌ DISAGREE"
print(f"Agreement: {status}")
print()
print("Paper claims length = 4  →",
      "✅ CONFIRMED" if r2['length'] == 4 else "❌ MISMATCH")

Paper example: A=[2,1,3,4,6,5,4], B=[1,2,3,5,6,4]
Baseline result:        seq=[1, 3, 6, 4], len=4
Match-sensitive result: seq=[2, 3, 6, 4], len=4
M (match count):        7
J (distinct j-ranks):   6,  R (distinct value-ranks): 6
Agreement: ✅ BOTH AGREE

Paper claims length = 4  → ✅ CONFIRMED


## ✅ Test 2: Agreement Test on 500 Random Pairs (Theorem 4 Verification)

Theorem 4 states that Algorithm 3 is **correct** — it always outputs the same LCBS length as the baseline (Algorithm 2). We verify this empirically across 500 random instances with varying sizes and alphabet sizes.

In [7]:
random.seed(42)
n_tests = 500
agree   = 0
disagree_cases = []

for t in range(n_tests):
    n     = random.randint(5, 30)
    m     = random.randint(5, 30)
    alpha = random.randint(3, 12)
    A_t   = [random.randint(1, alpha) for _ in range(n)]
    B_t   = [random.randint(1, alpha) for _ in range(m)]

    r1 = solve_lcbs_baseline(A_t, B_t)
    r2 = solve_lcbs_match_sensitive(A_t, B_t)

    if r1['length'] == r2['length']:
        agree += 1
    else:
        disagree_cases.append({'A': A_t, 'B': B_t,
                               'base_len': r1['length'], 'base_seq': r1['sequence'],
                               'ms_len':   r2['length'], 'ms_seq':   r2['sequence']})

print(f"Agreement test on {n_tests} random pairs:")
print(f"  Agreed:    {agree}/{n_tests}  ({100*agree/n_tests:.1f}%)")
print(f"  Disagreed: {len(disagree_cases)}/{n_tests}")

if not disagree_cases:
    print("\n✅ THEOREM 4 CONFIRMED: Both algorithms always return the same length.")
else:
    print("\n❌ Discrepancies found — first up to 3 cases:")
    for c in disagree_cases[:3]:
        print(f"  A={c['A']}")
        print(f"  B={c['B']}")
        print(f"  Baseline:       len={c['base_len']}, seq={c['base_seq']}")
        print(f"  Match-sensitive: len={c['ms_len']},  seq={c['ms_seq']}")
        print()

Agreement test on 500 random pairs:
  Agreed:    500/500  (100.0%)
  Disagreed: 0/500

✅ THEOREM 4 CONFIRMED: Both algorithms always return the same length.


## 📋 Notebook 2 Summary

### What was implemented

| Component | Paper Reference | Complexity |
|---|---|---|
| `build_match_set()` | Section 4.1 | O(n+m+M) |
| `coordinate_compress()` | Section 4.1 | O(M log M) |
| `DominanceMaxOracle` (2D BIT) | Lemma 4, Section 4.3 | O(log²M) per op |
| Forward pass — Eq. (2) | Section 4.2 | O(M log²M) |
| Backward pass — Eq. (3) + mirror | Section 4.2–4.4 | O(M log²M) |
| `solve_lcbs_match_sensitive()` — Algorithm 3 | Section 4, Theorem 4 & 5 | O(M log²M) total |

### What was verified

| Test | Result |
|---|---|
| Paper motivating example (length = 4) | ✅ Confirmed |
| Theorem 4: agreement with baseline on 500 random pairs | ✅ 500/500 (100%) |

### Key implementation notes

- The **mirrored $j$-rank** $\hat{x} = J - r_J(v) + 1$ is the critical trick that converts the backward-pass suffix query into a standard prefix query — handled identically by the same BIT structure.
- The **sparse dict-based BIT** avoids pre-allocating a full $J \times R$ array; only touched Fenwick cells are stored.
- Both passes share the same `DominanceMaxOracle` class — the only difference is the coordinate fed to `update`/`query`.

---